In [ ]:
from board import Loc, Target, Cell, Node, Board, Transformer, parse, DIGITS

In [ ]:
from collections.abc import Generator


def iter_layer(board: Board, digit: int) -> Generator[Target]:
    for node in board:
        if digit in node.cell:
            yield Target(node.loc, digit)

## A puzzle


In [ ]:
puzzle = parse("""
.8.....52
.......87
....98...
4...3.6..
.2.7.....
.........
6..8.2...
...5.91..
9........
""")

## UI


In [ ]:
from ipywidgets import widgets
from canvas import SudokuCanvas

canvas = SudokuCanvas()
canvas[2].global_alpha = 0.5
canvas.draw_grid()


layer_selecting = widgets.RadioButtons(options=(0,) + DIGITS, value=None)


def redraw_board():
    canvas.clear_highlights()
    canvas.draw_board(puzzle)


def highlight_layer(digit: int):
    canvas.clear_highlights()
    canvas.highlight_targets(iter_layer(puzzle, digit))


def on_select_layer(change):
    if change.new == 0:
        canvas.clear_highlights()
    else:
        highlight_layer(int(change.new))


layer_selecting.observe(on_select_layer, "value")


widgets.HBox(
    [
        canvas,
        layer_selecting,
    ],
    style=dict(justify_content="flex-start"),
)

In [ ]:
redraw_board()

## Solving

kinda


In [ ]:
def fillempty(node: Node):
    if node.cell.is_empty:
        return Node(node.loc, Cell(DIGITS))
    else:
        return node

In [ ]:
def cleanup(board: Board) -> Transformer:

    def finals(locs):
        cells = [n.cell for n in board.slice(locs)]
        return set(c.final for c in cells if c.is_final)

    def cleanup(node: Node) -> Node:
        if node.cell.is_final:
            return node
        blkfinals = finals(node.loc.allblk())
        rowfinals = finals(node.loc.allrow())
        colfinals = finals(node.loc.allcol())
        allfinals = blkfinals | rowfinals | colfinals
        return Node(node.loc, Cell(set(node.cell) - allfinals))

    return cleanup


In [ ]:
puzzle = Board.transform(puzzle, fillempty)
puzzle = Board.transform(puzzle, cleanup(puzzle))